In [ ]:
import json
import os
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rctoolbox.utils.project_root import find_project_root

project_root = find_project_root()
os.chdir(project_root)
print(f"Current working directory: {os.getcwd()}")

def reset_logging():
    import logging
    logging.root.handlers.clear()
    for logger in logging.Logger.manager.loggerDict.values():
        if hasattr(logger, "handlers"):
            logger.handlers.clear()

In [ ]:
HEMS_ID = "VUBC"

OUT_DIR_PATH = project_root / "examples" / "run_identification" / "six_r3c_eniso13790" / HEMS_ID
# print(f"Output dir: {OUT_DIR_PATH}")

## DATA

In [ ]:
DATA_DIR  = project_root / "examples" / "data"

data_path = DATA_DIR / f'{HEMS_ID}_data.pkl'
DATA = pd.read_pickle(data_path)
SEASON = 'w2324'
season = DATA[SEASON]
df = season['df']

print('season:        ', SEASON)
print('df shape:      ', df.shape)
print('time range:    ', df.index.min(), '->', df.index.max())
df.columns

In [ ]:
# Define time windows for training
from rctoolbox.data.time_windows_generator import generate_time_windows
from rctoolbox.data.numpydataloader import DataLoader as NumpyDataLoader

wdh = 3; spb = int(1e10); ssh=1;
pre_window_lag = timedelta(hours=24)

# t1_train = datetime(2023,11,1)
t1_train = datetime(2023,12,1)
t2_train = datetime(2024,2,1)
tws_train = generate_time_windows(start_time=t1_train, end_time=t2_train, window_duration_hours=wdh, samples_per_batch=spb, slide_step_hours=ssh)

tws = {"train": tws_train}
dataloader = NumpyDataLoader(df)
dataloader.load_all_data_batches(time_windows=tws, features=df.columns.tolist(), pre_window_lag=pre_window_lag)

In [ ]:
RESULTS_DIR_PATH = OUT_DIR_PATH / "results"
OUT_DIR_PATH.mkdir(parents=True, exist_ok=True)
RESULTS_DIR_PATH.mkdir(exist_ok=True)

from rctoolbox.models.rc.six_r3c_eniso13790.build_data_dict import build_data_dict
import joblib
data_joblib = {
    "train": build_data_dict(dataloader, "train", Phi_HC_col="Phi_HC", T_h_col="y2"),
    "val":   None,
}
data_joblib_path = OUT_DIR_PATH / "data_train.joblib"
joblib.dump(data_joblib, data_joblib_path)

## Identification
---

In [ ]:
from rctoolbox.utils.config_loader import load_default_building_config
building_config = load_default_building_config(HEMS_ID)
building_config

In [ ]:
W_TI = 0.95   # weight on the indoor-air error; the rest goes to the heating circuit


def weighted_temp_objective(y_true, y_pred):
    """Weighted RMSE over the two supervised states.

    y_true, y_pred : ndarray, shape (B, T, 2) — states [T_i, T_h] as stacked by
    ``TemperatureProblem`` with ``output_states=[[0, "T_i"], [1, "T_h"]]``.
    """
    rmse_Ti = float(np.sqrt(np.nanmean((y_true[..., 0] - y_pred[..., 0]) ** 2)))
    rmse_Th = float(np.sqrt(np.nanmean((y_true[..., 1] - y_pred[..., 1]) ** 2)))
    return W_TI * rmse_Ti + (1.0 - W_TI) * rmse_Th

In [ ]:
model_config = {
    "model_type":    "6R3C_temperature",
    "init_config":   {"method": "dynamic"},
    "output_states": [[0, "T_i"], [1, "T_h"]]
    }

# Custom objective: weighted RMSE on (T_i, T_h), defined in objectives.py next to this
# notebook and referenced here so that the CLI (rct-train) and this notebook share it.

optimization_config = {"objective_function": "__main__:weighted_temp_objective", "pop_size": 100, "n_gen": 100}

# selection_config = {"criteria": "train error", "select_R_star": False, "threshold_config": {"type":"percentage", "x":0.05}}
selection_config = {"criteria": "train error", "select_R_star": True, "threshold_config": {"type":"percentage", "x":0.05}}

metric = "RMSE"
custering_config = {"min_samples_ratio": 0.1, "cl_eval_metric": metric, "cl_eval_criteria": "train error"}

identification_config = {
    "model_config": model_config,
    "optimization_config": optimization_config,
    "selection_config": selection_config,
    "clustering_config": custering_config
}

identification_config_path  = OUT_DIR_PATH / "identification_config.json"
with open(identification_config_path, "w") as f:
    json.dump(identification_config, f, indent=4)

In [ ]:
# --- RUN PIPELINE ---

reset_logging()

from argparse import Namespace
from rctoolbox.pipeline.runner import run_pipeline

run_pipeline(
    Namespace(
        building_config=HEMS_ID,
        data_path=str(data_joblib_path),
        identification_config=str(identification_config_path),
        results_directory_path=str(RESULTS_DIR_PATH),
        seed=23, save_figures=True, save_optimizer=False, save_dfRtopN=True, stage="1-3"
    ),
)

## Evaluate
---

In [ ]:
# Load the identification results.
results_json_path = f"{OUT_DIR_PATH}/results/results.json"
with open(results_json_path, "r") as f:
    results = json.load(f)
results

In [ ]:
# Set up the model

from rctoolbox.models.loader import load_best_model
from rctoolbox.models.rc.six_r3c_eniso13790.model import SixR3C_ENISO13790

model = load_best_model(results_json_path, SixR3C_ENISO13790, False)
model.params

In [ ]:
## GENERATE TIME WINDOWS

from rctoolbox.data.time_windows_generator import generate_time_windows
from rctoolbox.data.numpydataloader import DataLoader as NumpyDataLoader

t1 = datetime(2023,11,1)
t2 = datetime(2024,4,1)

wdh = 24; ssh = wdh
tws = generate_time_windows(start_time=t1, end_time=t2, samples_per_batch=int(1e6), window_duration_hours=wdh, slide_step_hours=ssh)
tws_test = {"test": tws}

dataloader = NumpyDataLoader(df)
dataloader.load_all_data_batches(time_windows=tws_test, features=df.columns.tolist(), pre_window_lag=pre_window_lag)

In [ ]:
# RUN SIMULATION on the test batch.
from rctoolbox.evaluation.forecast_metrics import evaluate_forecast
from rctoolbox.evaluation.error_threshold import err_thr

batch = dataloader.batches["test"]["batch_0"]
raw_dfs = dataloader.raw_dfs["test"]["batch_0"]


di = batch["data_inputs"]

t       = batch["t_array"]
T_i     = di["y1"]
T_h     = di["y2"]
T_a     = di["T_a"]
Phi_sol = di["Phi_sol"]
Phi_HC  = di["Phi_HC"]


T_i_lag = di["y1_1Dlag"]
T_h_lag = di["y2_1Dlag"]
T_i0  = T_i[:, 0];  T_h0  = T_h[:, 0]
T_i0_ = T_i_lag[:, 0]; T_h0_ = T_h_lag[:, 0]
y_0  = np.stack([T_i0,  T_h0,  T_i0  - 1], axis=1)
y_0_assumption = y_0.copy()
y_0_ = np.stack([T_i0_, T_h0_, T_i0_ - 1], axis=1)

init_data = {
    "y_0":      y_0,
    "y_0_":     y_0_,
    "T_i_":     T_i_lag,
    "T_h_":     T_h_lag,
    "t_":       di["t_array_1Dlag"],
    "T_a_":     di["T_a_1Dlag"],
    "Phi_sol_": di["Phi_sol_1Dlag"],
    "Phi_HC_":  di["Phi_HC_1Dlag"],
}
y_0 = model.get_initial_state(init_data)

y_pred = model.forecast_temperature(y_0, t, T_a, Phi_sol, Phi_HC)
T_i_pred = y_pred[:, :, 0]
T_h_pred = y_pred[:, :, 1]
T_e_pred = y_pred[:, :, 2]

if "missing_flag" in di:
    mf = di["missing_flag"] == 1
    T_i_pred[mf] = np.nan
    T_h_pred[mf] = np.nan

# Per-state evaluation (drop t=0 because that's the boundary used for init).
p_Ti = evaluate_forecast(T_i[:, 1:].copy(), T_i_pred[:, 1:].copy(), error_thresholds=err_thr)
p_Th = evaluate_forecast(T_h[:, 1:].copy(), T_h_pred[:, 1:].copy(), error_thresholds=err_thr)

batch_dfs = raw_dfs.copy()
for i, (sampled_idx, bdf) in enumerate(batch_dfs.items()):
    bdf["T_i_pred"] = T_i_pred[i]
    bdf["T_h_pred"] = T_h_pred[i]
    bdf["T_e_pred"] = T_e_pred[i]

dfbs = pd.concat([df_ for df_ in batch_dfs.values()], axis=0)
# dfbs = dfbs[~dfbs.index.duplicated(keep='first')]

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(12, 6), dpi=100, sharex=True)
ax0, ax1 = axs
ax0r, ax1r = ax0.twinx(), ax1.twinx()

dsd = "default"
dssp = "steps-post"

t1 = datetime(2023, 11, 1)
t1 = datetime(2024, 2, 1)
t2 = t1 + timedelta(days=5)
dfs = dfbs.loc[t1:t2]

dfs[["y1", "T_i_pred"]].plot(ax=ax0, drawstyle=dsd, lw=2)
dfs[["y1_set"]].plot(ax=ax0r, drawstyle=dssp, lw=1, ls="--", c="blue")
dfs[["y2", "T_h_pred"]].plot(ax=ax1, drawstyle=dsd, lw=2)  
dfs[["Phi_HC"]].plot(ax=ax1r, drawstyle=dssp, lw=0.5, ls="--", c="darkred")

ax0.set_ylabel("T_i [°C]")
ax1.set_ylabel("T_h [°C]")
ax1r.set_ylabel("Phi_HC [W]")
ax0.legend(loc="upper left")
ax1.legend(loc="upper left")
plt.tight_layout()

In [ ]:
from rctoolbox.evaluation.forecast_metrics import organise_results_in_dataframe
print("--- T_i ---")
print(organise_results_in_dataframe(p_Ti["summary_statistics"], thresholds=p_Ti["threshold_compliance_%"]))
print("\n--- T_h ---")
print(organise_results_in_dataframe(p_Th["summary_statistics"], thresholds=p_Th["threshold_compliance_%"]))

### Forecast with uncertainty band

Load `dfRtopN` (the identified final parameter space $\mathcal{R}^*$), subsample $M$ candidates, and forecast with each. The median forecast is the point prediction; the p05–p95 spread across the $M$ ensemble members is the uncertainty band. Same time window and axes as the single-model plot above.

Subsampling is used to keep runtime reasonable — the full $\mathcal{R}^*$ can contain thousands of candidates, but $M \approx 30\text{–}50$ already produces a visually smooth band (the Numba-JIT'd 6R3C simulator runs each forecast in tens of milliseconds).

In [ ]:
# --- Ensemble forecast from dfRtopN (final parameter space R*) ---
import ast

M         = 200    # ensemble size; bump for smoother band, drop for faster runtime
LO_PCT    = 5     # lower percentile of the uncertainty band
HI_PCT    = 95    # upper percentile of the uncertainty band
SEED      = 23

dfRtopN_path = RESULTS_DIR_PATH / "dfRtopN.csv"
dfRtopN = pd.read_csv(dfRtopN_path)
# `params` column is a dict serialized as a string; parse back into real dicts
dfRtopN["params"] = dfRtopN["params"].apply(ast.literal_eval)

# Uniform subsample without replacement
rng = np.random.default_rng(SEED)
n_pool = len(dfRtopN)
M_eff = min(M, n_pool)
sample_rows = dfRtopN.iloc[rng.choice(n_pool, size=M_eff, replace=False)]
print(f"Sampled {M_eff} models from {n_pool}-row dfRtopN "
      f"(pool spans train_error {dfRtopN['train_error'].min():.4f} → {dfRtopN['train_error'].max():.4f})")

# Forecast each sampled model on the same test batch used in the cell above.
# init_data, t, T_a, Phi_sol, Phi_HC, T_i_pred come from the previous cell.
T_i_ens = np.empty((M_eff, *T_i_pred.shape), dtype=float)
T_h_ens = np.empty((M_eff, *T_h_pred.shape), dtype=float)

for m, (_, row) in enumerate(sample_rows.iterrows()):
    mdl = SixR3C_ENISO13790()
    if model_config["init_config"] is not None:
        mdl.set_initialization(init_config=model_config["init_config"])
    mdl.set_params(row["params"])
    y0_m = mdl.get_initial_state(init_data)
    yp_m = mdl.forecast_temperature(y0_m, t, T_a, Phi_sol, Phi_HC)
    T_i_ens[m] = yp_m[:, :, 0]
    T_h_ens[m] = yp_m[:, :, 1]

# Mirror the missing-value mask handling from the single-model cell
if "missing_flag" in di:
    mf = di["missing_flag"] == 1
    T_i_ens[:, mf] = np.nan
    T_h_ens[:, mf] = np.nan

# Reduce across ensemble axis → median + percentile band
T_i_med = np.nanmedian(T_i_ens, axis=0)
T_i_lo  = np.nanpercentile(T_i_ens, LO_PCT, axis=0)
T_i_hi  = np.nanpercentile(T_i_ens, HI_PCT, axis=0)
T_h_med = np.nanmedian(T_h_ens, axis=0)
T_h_lo  = np.nanpercentile(T_h_ens, LO_PCT, axis=0)
T_h_hi  = np.nanpercentile(T_h_ens, HI_PCT, axis=0)

# Assemble a continuous time-indexed frame the same way the single-model cell does
batch_dfs_ens = {k: v.copy() for k, v in raw_dfs.items()}
for i, (_, bdf) in enumerate(batch_dfs_ens.items()):
    bdf["T_i_pred_med"] = T_i_med[i]
    bdf["T_i_pred_lo"]  = T_i_lo[i]
    bdf["T_i_pred_hi"]  = T_i_hi[i]
    bdf["T_h_pred_med"] = T_h_med[i]
    bdf["T_h_pred_lo"]  = T_h_lo[i]
    bdf["T_h_pred_hi"]  = T_h_hi[i]
dfbs_ens = pd.concat(list(batch_dfs_ens.values()), axis=0)

# --- Plot: same layout as the single-model cell above, with p05–p95 band ---
fig, axs = plt.subplots(2, 1, figsize=(12, 6), dpi=150, sharex=True)
ax0, ax1 = axs
ax0r, ax1r = ax0.twinx(), ax1.twinx()

t1_plot = datetime(2024, 2, 1)
t2_plot = t1_plot + timedelta(days=5)

# t1_plot = datetime(2023, 12, 11)
# t2_plot = t1_plot + timedelta(days=2)


dfs = dfbs_ens.loc[t1_plot:t2_plot]

# Measured (black)
ax0.plot(dfs.index, dfs["y1"], color="black", lw=1.6, label="$T_i$ (measured)")
ax1.plot(dfs.index, dfs["y2"], color="black", lw=1.6, label="$T_h$ (measured)")

# Ensemble median (orange) + p05–p95 band
ax0.plot(dfs.index, dfs["T_i_pred_med"], color="tab:orange", lw=1.8, label="$T_i$ pred (median)")
ax0.fill_between(dfs.index, dfs["T_i_pred_lo"], dfs["T_i_pred_hi"],
                 color="tab:orange", alpha=0.25, linewidth=0,
                 label=f"$T_i$ pred (p{LO_PCT}–p{HI_PCT})")

ax1.plot(dfs.index, dfs["T_h_pred_med"], color="tab:orange", lw=1.8, label="$T_h$ pred (median)")
ax1.fill_between(dfs.index, dfs["T_h_pred_lo"], dfs["T_h_pred_hi"],
                 color="tab:orange", alpha=0.25, linewidth=0,
                 label=f"$T_h$ pred (p{LO_PCT}–p{HI_PCT})")

# Setpoint / heat-power on right axes
dfs[["y1_set"]].plot(ax=ax0r, drawstyle="steps-post", lw=1, ls="--", c="blue")
dfs[["Phi_HC"]].plot(ax=ax1r, drawstyle="steps-post", lw=0.5, ls="--", c="darkred")

ax0.set_ylabel("$T_i$ [°C]")
ax1.set_ylabel("$T_h$ [°C]")
ax0r.set_ylabel("$T_{set}$ [°C]", color="blue")
ax1r.set_ylabel("$\\Phi_{HC}$ [W]", color="darkred")
ax0.set_title(f"6R3C forecast with uncertainty band  ·  M = {M_eff}  ·  VUBC, {t1_plot.date()} → {t2_plot.date()}")

ax0.legend(loc="upper left", fontsize=9)
ax1.legend(loc="upper left", fontsize=9)
for a in (ax0, ax1):
    a.grid(True, ls="--", lw=0.5, alpha=0.5)

plt.tight_layout()

# --- Save for README ---
readme_fig_dir = project_root / "examples" / "figures"
readme_fig_dir.mkdir(parents=True, exist_ok=True)
readme_fig_path = readme_fig_dir / "six_r3c_forecast_with_uncertainty.png"
plt.savefig(readme_fig_path, dpi=200, bbox_inches="tight")
print(f"Saved: {readme_fig_path.relative_to(project_root)}")

plt.show()

## Predict for control events
---

In [ ]:
with open("examples/data/Control_Experiments/DR_VUBC_w2324_UTC.json") as f:
    DRs_dict = json.load(f)

DR_COL = "DR"  # DR < 0 = setback, DR == 0 = comfort, DR > 0 = boost

print(f"DR experiments in w2324: {len(DRs_dict)}")
print(list(DRs_dict.keys()))

In [ ]:
from rctoolbox.data.time_windows_generator import generate_time_windows
from rctoolbox.data.numpydataloader import DataLoader as NumpyDataLoader

from zoneinfo import ZoneInfo
TZ_UTC   = ZoneInfo("UTC")
TZ_LOCAL = ZoneInfo("Europe/Ljubljana")

wdh = 24; ssh = 1
extended_wdh = 24*4
pre_window_lag = timedelta(hours=24)

dr_tws = {}  # dr_date -> tws_exp  {batch_0: {sample_0, sample_1, ...}}

for dr_date, dr_info in DRs_dict.items():
    t1 = datetime.fromisoformat(dr_info["time-window"][0]).replace(tzinfo=TZ_UTC).astimezone(TZ_LOCAL).replace(tzinfo=None)
    t2 = datetime.fromisoformat(dr_info["time-window"][1]).replace(tzinfo=TZ_UTC).astimezone(TZ_LOCAL).replace(tzinfo=None)

    while t2 - t1 < timedelta(hours=extended_wdh):
        t1 -= timedelta(hours=1)
        if t2 - t1 >= timedelta(hours=extended_wdh): break
        t2 += timedelta(hours=1)

    dr_tws[dr_date] = generate_time_windows(
        start_time=t1, end_time=t2,
        samples_per_batch=int(1e6), window_duration_hours=wdh, slide_step_hours=ssh
    )

for dr_date, tws_exp in dr_tws.items():
    n_samples = sum(len(b) for b in tws_exp.values())
    print(f"{dr_date}: {n_samples} sample(s)")

In [ ]:
from rctoolbox.models.rc.six_r3c_eniso13790.predict_functions import predict_for_batch

sim_results = {}

for dr_date, tws_exp in dr_tws.items():
    tws_test   = {"test": tws_exp}
    dataloader = NumpyDataLoader(df)
    dataloader.load_all_data_batches(time_windows=tws_test, features=df.columns.tolist(),
                                     pre_window_lag=pre_window_lag)
    data_dict = build_data_dict(dataloader, "test", Phi_HC_col="Phi_HC", T_h_col="y2")
    batch   = dataloader.batches["test"]["batch_0"]
    raw_dfs = dataloader.raw_dfs["test"]["batch_0"]

    _, batch_dfs = predict_for_batch(model, batch, raw_dfs)

    sim_results[dr_date] = {"bdfs": batch_dfs}

print("Simulation complete.")

In [ ]:
ncols = 4
nrows = int(np.ceil(len(DRs_dict) / ncols))
fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20, 3 * nrows), dpi=200)
axs = axs.flatten()

drawstyle = "default"

for ax_idx, (dr_date, result) in enumerate(sim_results.items()):
    ax  = axs[ax_idx]
    axr = ax.twinx()

    batch_dfs = result["bdfs"]

    # Full measured period spanning all samples
    t1_plot = min(sdf.index.min() for sdf in batch_dfs.values())
    t2_plot = max(sdf.index.max() for sdf in batch_dfs.values())

    df.loc[t1_plot:t2_plot, "y1"].plot(ax=ax, color="tab:blue", linewidth=2, alpha=1, label="y1", drawstyle=drawstyle)
    # df.loc[t1_plot:t2_plot, "y2"].plot(ax=ax, color="navy", linewidth=2, alpha=1, label="y2", drawstyle=drawstyle)
    (df.loc[t1_plot:t2_plot, "Phi_HC"] / 1e3).plot(ax=axr, color="k", linewidth=1, alpha=0.5, label="Phi_HC (measured)", drawstyle="steps-post")
    

    # Each sample prediction as a separate line
    for s_idx, (sample_key, sample_df) in enumerate(batch_dfs.items()):
        label = "Phi_HC_pred" if s_idx == 0 else None
        sample_df["T_i_pred"].plot(ax=ax, color="tab:orange", linewidth=1, alpha=0.7, linestyle="--", label=label, drawstyle=drawstyle)
        # sample_df["T_h_pred"].plot(ax=ax, color="tab:orange", linewidth=1, alpha=0.7, linestyle="--", label=label, drawstyle=drawstyle)

    ax.set_title(dr_date, fontsize=15)
    ax.set_xlabel("")

# Deduplicate legend on first axis
handles, labels = axs[0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
axs[0].legend(by_label.values(), by_label.keys(), fontsize=7, loc="upper left")

for ax in axs[len(DRs_dict):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
DRs_dict.keys()

dr_keys = ['2024-01-12', '2024-03-15', '2024-03-18']

### Per-event figures (T_i on top, T_h + Phi_HC on bottom)

For an arbitrary selection of DR events, plot each as its own 2×1 figure:
- **top**: measured indoor temperature `y1` (T_i) and the rolling T_i prediction,
- **bottom**: measured supply temperature `y2` (T_h) and the rolling T_h prediction on the left axis, measured `Phi_HC` on the right axis.

The next cell then re-uses the same helper to compose the selected events into a single combined figure (2 rows × N cols) and saves it.

In [ ]:
def plot_dr_event(
    dr_date, sim_result, df, ax_top, ax_bot,
    show_left_ylabels=True, show_right_ylabel=True, show_title=True,
):
    """Plot one DR event onto (ax_top, ax_bot). Returns the twin (right) axis of ax_bot."""
    batch_dfs = sim_result["bdfs"]

    t1_plot = min(sdf.index.min() for sdf in batch_dfs.values())
    t2_plot = max(sdf.index.max() for sdf in batch_dfs.values())

    # --- top: T_i measured + rolling T_i predictions ---
    df.loc[t1_plot:t2_plot, "y1"].plot(
        ax=ax_top, color="tab:blue", linewidth=2, label="T_i (meas)"
    )
    for s_idx, sample_df in enumerate(batch_dfs.values()):
        # "_nolegend_" hides subsequent samples from the legend; label=None would
        # fall back to Series.name ("T_i_pred") and produce a duplicate entry.
        label = "T_i_pred (rolling)" if s_idx == 0 else "_nolegend_"
        sample_df["T_i_pred"].plot(
            ax=ax_top, color="tab:orange", linewidth=1, alpha=0.5, linestyle="--", label=label
        )

    # --- bottom: T_h measured + rolling T_h predictions (left), Phi_HC (right) ---        
    ax_bot_r = ax_bot.twinx()
    (df.loc[t1_plot:t2_plot, "Phi_HC"] / 1e3).plot(
        ax=ax_bot_r, color="red", linewidth=1, alpha=1,
        drawstyle="steps-post", label="Phi_HC [kW]", zorder=0
    )

    df.loc[t1_plot:t2_plot, "y2"].plot(
        ax=ax_bot, color="navy", linewidth=2, label="T_h (meas)", zorder=10
    )
    for s_idx, sample_df in enumerate(batch_dfs.values()):
        label = "T_h_pred (rolling)" if s_idx == 0 else "_nolegend_"
        sample_df["T_h_pred"].plot(
            ax=ax_bot, color="tab:orange", linewidth=1, alpha=0.5, linestyle="--", label=label, zorder=10
        )

    # ax_top.set_ylabel("T_i [°C]" if show_left_ylabels else "")
    ax_top.set_ylabel("T_i [°C]" if show_left_ylabels else "", color="tab:blue")
    ax_top.tick_params(axis="y", colors="tab:blue")     # ticks + tick labels
    ax_top.spines["left"].set_color("tab:blue")         # left spine

    ax_bot.set_ylabel("T_h [°C]" if show_left_ylabels else "", color="navy")
    ax_bot.tick_params(axis="y", colors="navy")
    ax_bot.spines["left"].set_color("navy")

    ax_bot_r.set_ylabel("Phi_HC [kW]" if show_right_ylabel else "", color="red")
    ax_bot_r.tick_params(axis="y", colors="red")
    ax_bot_r.spines["right"].set_color("red")

    if show_title:
        ax_top.set_title(dr_date, fontsize=11)
    ax_top.set_xlabel("")
    ax_bot.set_xlabel("")

    # for ax in [ax_top, ax_bot]: ax.grid(which="both", ls="-", lw=0.5, alpha=0.5)
    return ax_bot_r


def _dedup_legend(ax, *, fontsize=7, loc="upper left"):
    h, l = ax.get_legend_handles_labels()
    by_label = dict(zip(l, h))
    ax.legend(by_label.values(), by_label.keys(), fontsize=fontsize, loc=loc)


# --- select an arbitrary N of DR events to plot ---
dr_keys = list(sim_results.keys())[:4]   # <- edit this to pick any subset
dr_keys = ['2024-01-12', '2024-03-15', '2024-03-18']
print(f"Plotting {len(dr_keys)} DR events: {dr_keys}")

for dr_date in dr_keys:
    fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(6, 5), dpi=100, sharex=True)
    ax_bot_r = plot_dr_event(dr_date, sim_results[dr_date], df, ax_top, ax_bot)

    _dedup_legend(ax_top)
    _dedup_legend(ax_bot)
    _dedup_legend(ax_bot_r, loc="upper right")

    plt.tight_layout()
    plt.show()

In [ ]:
# --- combined figure: same dr_keys laid out in one row (2 rows x N cols) ---
n = len(dr_keys)
fig, axs = plt.subplots(
    nrows=2, ncols=n,
    figsize=(4.5 * n, 6),
    dpi=150,
    sharex="col",
    sharey="row",     # left y-axes shared across each row
    squeeze=False,
)

right_axes = []
for col, dr_date in enumerate(dr_keys):
    ax_top = axs[0, col]
    ax_bot = axs[1, col]
    ax_bot_r = plot_dr_event(
        dr_date, sim_results[dr_date], df, ax_top, ax_bot,
        show_left_ylabels=(col == 0),     # only leftmost column keeps left-axis labels
        show_right_ylabel=(col == n - 1), # only rightmost column shows the Phi_HC label
        show_title=True,
    )
    right_axes.append(ax_bot_r)
    # remove per-panel auto-legends from ALL three axes (left, right, top);
    # we'll place a single deduplicated legend on the leftmost / rightmost below
    for a in (ax_top, ax_bot, ax_bot_r):
        leg = a.get_legend()
        if leg is not None:
            leg.remove()

# align right y-axes across the bottom row (twinx isn't captured by sharey='row')
r_ymin = min(rax.get_ylim()[0] for rax in right_axes)
r_ymax = max(rax.get_ylim()[1] for rax in right_axes)
for rax in right_axes:
    rax.set_ylim(r_ymin, r_ymax)

# hide inner-column tick labels & tick marks so shared axes don't repeat
for col in range(n):
    if col > 0:
        axs[0, col].tick_params(axis="y", left=False, labelleft=False)
        axs[1, col].tick_params(axis="y", left=False, labelleft=False)
    if col < n - 1:
        right_axes[col].tick_params(axis="y", right=False, labelright=False)

# single shared legend on the leftmost column (left-axis series)
_dedup_legend(axs[0, 0], fontsize=8)
_dedup_legend(axs[1, 0], fontsize=8)
# single shared legend on the rightmost column's right axis (Phi_HC)
_dedup_legend(right_axes[-1], fontsize=8, loc="upper right")

plt.tight_layout()
fig.subplots_adjust(wspace=0.05)   # minimize horizontal gap between columns

# --- save ---
combined_out_path = project_root / "examples/figures/six_r3c_iso_forecast.pdf"
combined_out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(combined_out_path, bbox_inches="tight")
fig.savefig(combined_out_path.with_suffix(".png"), bbox_inches="tight", dpi=200)
# print(f"Saved: {combined_out_path}")
# print(f"Saved: {combined_out_path.with_suffix('.png')}")

plt.show()